# Полные предложения с подвохами на больших моделях

Прогон `scripts/sentences_tricks.py` (протокол записан в начале файла и зафиксирован в системе контроля версий до прогона) на GPT-2 XL, Qwen2.5-7B, Qwen3-8B и Qwen3-14B, две диагностики и указатель первого порядка (`scripts/pointer_sentences.py`). Интерпретатор здесь не обучается: его веса лежат в репозитории (`results/sentences_tricks_interpreter.pt`).

Результаты авторских прогонов уже лежат в `results/sentences_tricks_colab_*.json`; ячейки ниже пишут свои файлы `results/my_*.json` и ничего не перезаписывают.

**Перед запуском:** «Среда выполнения» → «Сменить среду выполнения» → GPU (T4).

In [ ]:
!nvidia-smi

Код скачивается прямо из репозитория, вместе с весами интерпретатора.

In [ ]:
!rm -rf whybox && git clone -q https://github.com/knbww/whybox.git
%cd whybox

In [ ]:
!pip install -q "transformers>=4.56" pyyaml pytest bitsandbytes accelerate

Проверка, что предложения грамматичны до и после каждого подавления.

In [ ]:
!python -m pytest -q tests/test_tricks.py

## GPT-2 XL, 1,5 млрд параметров (float32)

Самая большая GPT-2. Около 10 минут.

In [ ]:
!PYTHONPATH=.:src:scripts python scripts/sentences_tricks.py \
    --models openai-community/gpt2-xl \
    --device cuda --dtype float32 \
    --out results/my_gpt2xl.json

## Qwen3-8B (4 бита)

Новое поколение Qwen. С градиентами в T4 помещается только в 4-битном виде (поправка 1 к протоколу); хватает ли точности, покажет ошибка полноты в выводе. Скачивание около 16 ГБ, всего около 20 минут.

In [ ]:
!PYTHONPATH=.:src:scripts python scripts/sentences_tricks.py \
    --models Qwen/Qwen3-8B \
    --device cuda --load-in-4bit \
    --out results/my_qwen3_8b.json

## Qwen3-14B (4 бита)

Самая большая модель, которая может поместиться в бесплатную T4. Скачивание около 30 ГБ, всего около 30–40 минут. Если не хватит памяти, ячейка упадёт с ошибкой CUDA out of memory, а результаты предыдущих ячеек останутся.

In [ ]:
!PYTHONPATH=.:src:scripts python scripts/sentences_tricks.py \
    --models Qwen/Qwen3-14B \
    --device cuda --load-in-4bit \
    --out results/my_qwen3_14b.json

## Проверка 1: Qwen2.5-7B (4 бита)

Чтение слабеет у больших Qwen3. Это то же семейство, что Qwen2.5 0,5–3B, которые читались без сжатия. Если Qwen2.5-7B прочитается так же хорошо, как они, дело не в 4-битной загрузке. Около 20 минут.

In [ ]:
!PYTHONPATH=.:src:scripts python scripts/sentences_tricks.py \
    --models Qwen/Qwen2.5-7B-Instruct \
    --device cuda --load-in-4bit \
    --out results/my_qwen25_7b.json

## Проверка 2: Qwen3-8B с 64 отрезками пути вместо 16

Это диагностика пути по поправке 1 к протоколу: в итоговые выводы она не входит, в файле результата помечена как `diagnostic`. Сравнить с прогоном на 16 отрезках (`sentences_tricks_colab_qwen3_8b.json`): если ошибка полноты заметно упадёт, а точность вырастет, чтение слабело из-за неточного сигнала, а не из-за интерпретатора. Около 30–45 минут.

In [ ]:
!PYTHONPATH=.:src:scripts python scripts/sentences_tricks.py \
    --models Qwen/Qwen3-8B \
    --device cuda --load-in-4bit --steps 64 \
    --out results/my_qwen3_8b_steps64.json

## Проверка 3: Qwen3-14B с 64 отрезками пути вместо 16

Та же диагностика по поправке 1, на самой большой модели, где чтение ослабло сильнее всего. В итоговые выводы не входит. Сравнить с прогоном на 16 отрезках (`sentences_tricks_colab_qwen3_14b.json`). Память та же, что на 16 отрезках: точки пути считаются по очереди. Около 45–50 минут.

In [ ]:
!PYTHONPATH=.:src:scripts python scripts/sentences_tricks.py \
    --models Qwen/Qwen3-14B \
    --device cuda --load-in-4bit --steps 64 \
    --out results/my_qwen3_14b_steps64.json

## Указатель первого порядка на семи моделях (протокол `scripts/pointer_sentences.py`)

Сумма интегрированных градиентов по позициям каждой причины, рядом с интерпретатором, и точность интерпретатора там, где указатель ошибается. Те же предложения и настройки, что в основных прогонах. Около 60 минут на все три ячейки.

In [ ]:
!PYTHONPATH=.:src:scripts python scripts/pointer_sentences.py \
    --models gpt2-large Qwen/Qwen2.5-1.5B-Instruct openai-community/gpt2-xl \
    --device cuda --dtype float32 \
    --out results/my_pointer_fp32.json

In [ ]:
!PYTHONPATH=.:src:scripts python scripts/pointer_sentences.py \
    --models Qwen/Qwen2.5-3B-Instruct \
    --device cuda --dtype float16 \
    --out results/my_pointer_fp16.json

In [ ]:
!PYTHONPATH=.:src:scripts python scripts/pointer_sentences.py \
    --models Qwen/Qwen2.5-7B-Instruct Qwen/Qwen3-8B Qwen/Qwen3-14B \
    --device cuda --load-in-4bit \
    --out results/my_pointer_4bit.json

## Итог и скачивание результатов

In [ ]:
import glob, json
from google.colab import files
for f in sorted(glob.glob('results/my_*.json')):
    d = json.load(open(f))
    if 'pooled' in d:
        for model, by_t in d['results'].items():
            for t, r in by_t.items():
                if not r['control']['passed']:
                    print(f"{model:30s} {t:8s} CONTROL FAILED {r['control']['failures']}")
                    continue
                print(f"{model:30s} {t:8s} interpreter {r['interpreter']['ALL']:.3f}  pointer {r['pointer']['ALL']:.3f}  "
                      f"interpreter where pointer wrong {r['interp_where_pointer_wrong']['ALL']:.3f} (n {r['n_pointer_wrong']['ALL']})")
    else:
        tag = '' if d['provenance'].get('role', 'read-out') == 'read-out' else '  [диагностика, шагов ' + str(d['provenance'].get('steps')) + ']'
        for model, by_t in d['results'].items():
            for t, r in by_t.items():
                acc = {k: round(v, 3) for k, v in r['arms']['interpreter'].items() if v == v}
                print(f"{model:30s} {t:8s} {r['read_out'] + tag:26s} interpreter {acc}  "
                      f"live {r['arms']['live']['ALL']:.3f}  completeness {r['completeness_residual'][0]:.4f}")
    files.download(f)
